In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from tqdm import tqdm
import time
from pathlib import Path

In [ ]:
trace_num = 2000
order = '3'
opt = 'o3'  # o0 or o3
scope = ''  # _chipwhisperer or space
algo = 'PQClean'  # orgPoly or PQClean or pqm4
platform = 'CW308_STM32F4'  # CW308_STM32F3 or CW308_STM32F4 or CWLITEARM

# 構建檔案路徑
trace_path = f'traces{scope}/{trace_num}/traces_unfixed_message_{trace_num}_{platform}_{opt}_{algo}_{order}'

t = np.load(f'{trace_path}/trace.npy')
m = np.load(f'{trace_path}/message.npy')

In [ ]:
print(t.shape)
print(m.shape)
print(m)

In [ ]:
# 參數設定
fs = 500e6          # 採樣頻率 (Hz)
cutoff_freq = 700e3   # 截止頻率 (Hz)
filter_order = 5   # 濾波器階數

# 計算正規化截止頻率
nyquist = fs / 2
normal_cutoff = cutoff_freq / nyquist

# 顯示巴特沃斯濾波器參數
print(f"  巴特沃斯濾波器參數:")
print(f"  採樣頻率: {fs:.0f} Hz")
print(f"  截止頻率: {cutoff_freq:.0f} Hz")
print(f"  濾波器階數: {filter_order}")
print(f"  正規化截止頻率: {normal_cutoff:.3f}")

In [ ]:
# 設計二階節 (Second-order sections)
sos = butter(filter_order, normal_cutoff, btype='low', analog=False, output='sos')


In [ ]:
# 準備輸出陣列
filtered_traces_array = np.zeros_like(t)
failed_count = 0

# 進度條過濾處理
for i in tqdm(range(len(t)), desc="濾波處理 (SOS)"):
    try:
        # 雙向濾波以減少相位失真
        filtered_traces_array[i] = sosfiltfilt(sos, t[i])
    except Exception as e:
        print(f"❌ 軌跡 {i} 處理失敗: {e}")
        failed_count += 1
        # 如果失敗，就保留原始軌跡
        filtered_traces_array[i] = t[i]

print(f"\n✅ 濾波處理完成!")
print(f"  成功處理: {len(t) - failed_count} 條")
print(f"  處理失敗: {failed_count} 條")

In [ ]:
filter_name = 'Butterworth_lowpass'
cutoff_freq_MHz = int(cutoff_freq / 1e3)  # 轉換為整數 7
output_dir = f'traces_{filter_name}/cf{cutoff_freq_MHz}K_order{filter_order}'
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

# 保存濾波後的軌跡
trace_output_file = output_path / "trace.npy"
np.save(trace_output_file, filtered_traces_array)
print(f"✅ 濾波軌跡已保存: {trace_output_file}")

# 保存對應的訊息數據
message_output_file = output_path / "message.npy"
np.save(message_output_file, m)
print(f"✅ 訊息數據已保存: {message_output_file}")
print(filtered_traces_array.shape)
print(m.shape)

In [ ]:
info_file = output_path / "filter_info.txt"
with open(info_file, 'w', encoding='utf-8') as f:
    f.write(f"Chebyshev Type I 濾波器處理資訊\n")
    f.write(f"========================\n")
    f.write(f"原始數據路徑: {trace_path}\n")
    f.write(f"濾波器類型: Butterworth 低通濾波器\n")
    f.write(f"截止頻率: {cutoff_freq} Hz\n")
    f.write(f"濾波器階數: {filter_order}\n")
    f.write(f"採樣頻率: {fs} Hz\n")
    f.write(f"原始軌跡數: {len(t)}\n")
    f.write(f"濾波軌跡數: {len(filtered_traces_array)}\n")

print(f"✅ 濾波器資訊已保存: {info_file}")
print(f"📁 所有檔案已保存至: {output_path}")

In [ ]:
print(filtered_traces_array[0])
print(m)

#### 濾波後單條trace

In [ ]:
plt.figure(figsize=(16, 8))
plt.rc('xtick', labelsize=20)   # x 軸刻度字體大小
plt.rc('ytick', labelsize=20)   # y 軸刻度字體大小
plt.xlabel('Sample Points', fontsize=20)
plt.ylabel('Power Consumption', fontsize=20)
plt.plot(filtered_traces_array[0])

# plt.savefig(f'figure/trace_lowpass_filter_remove_noise/PQClean_unfixed_message_{trace_num}_single_trace_butterworth_filter.png', dpi=200, bbox_inches='tight')

#### 濾波後平均

In [ ]:
plt.figure(figsize=(16, 8))
plt.rc('xtick', labelsize=20)   # x 軸刻度字體大小
plt.rc('ytick', labelsize=20)   # y 軸刻度字體大小
plt.xlabel('Sample Points', fontsize=20)
plt.ylabel('Power Consumption', fontsize=20)
plt.plot(np.mean(filtered_traces_array, axis=0))
# plt.plot(t_filter[0])

# plt.savefig(f'figure/trace_lowpass_filter_remove_noise/PQClean_unfixed_message_{trace_num}_mean_trace_butterworth_filter.png', dpi=200, bbox_inches='tight')

#### 濾波後比較

In [ ]:
plt.figure(figsize=(16, 24))

plt.subplot(3, 1, 1)
plt.plot(t[0], color='gray')
plt.title('Original Trace', fontsize=20)
plt.rc('xtick', labelsize=20)   # x 軸刻度字體大小
plt.rc('ytick', labelsize=20)   # y 軸刻度字體大小
plt.xlabel('Sample Points', fontsize=20)
plt.ylabel('Power Consumption', fontsize=20)

plt.subplot(3, 1, 2)
plt.plot(filtered_traces_array[0])
plt.title('Chebyshev Type I Filter', fontsize=20)
plt.rc('xtick', labelsize=20)   # x 軸刻度字體大小
plt.rc('ytick', labelsize=20)   # y 軸刻度字體大小
plt.xlabel('Sample Points', fontsize=20)
plt.ylabel('Power Consumption', fontsize=20)

plt.subplot(3, 1, 3)
plt.plot(t[0])
plt.plot(filtered_traces_array[0])
plt.title('Difference', fontsize=20)
plt.rc('xtick', labelsize=20)   # x 軸刻度字體大小
plt.rc('ytick', labelsize=20)   # y 軸刻度字體大小
plt.xlabel('Sample Points', fontsize=20)
plt.ylabel('Power Consumption', fontsize=20)


# plt.savefig(f'figure/trace_lowpass_filter_remove_noise/PQClean_unfixed_message_{trace_num}_single_trace_compare.png', dpi=200, bbox_inches='tight')